In [1]:
CFG = {'seed': 0, 'holdout': 2000, 'w': (64, 128, 256), 'nb': (3, 3, 3), 'drop': 0.1, 'ema': 0.999, 'epochs': 80, 'bs': 256, 'lr': 0.2, 'wd': 0.0005, 'ls': 0.1, 'rob': 0.8, 'erase': 0.4, 'cutmix': 0.5, 'inorm': True}
import os, glob, time, math, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
dev = 'cuda'
torch.backends.cudnn.benchmark = True
random_seed = CFG['seed']; torch.manual_seed(random_seed); np.random.seed(random_seed)

# ---------- data (competition files only) ----------
R = os.path.dirname(glob.glob('/kaggle/input/**/train_labels.csv', recursive=True)[0])
tr = pd.read_csv(R + '/train_labels.csv'); te = pd.read_csv(R + '/test.csv'); ss = pd.read_csv(R + '/sample_submission.csv')
print(tr.shape, te.shape, ss.shape, list(tr.columns), list(ss.columns))
lc = 'label'; fcol = 'filename'
def resolve(d, v):
    for e in ['', '.png', '.jpg', '.jpeg']:
        p = f'{R}/{d}/{v}{e}'
        if os.path.exists(p): return lambda x, d=d, e=e: f'{R}/{d}/{x}{e}'
    raise FileNotFoundError((d, v))
def load(d, ids):
    f = resolve(d, ids[0])
    with ThreadPoolExecutor(16) as ex:
        return np.stack(list(ex.map(lambda i: np.array(Image.open(f(i)).convert('RGB')), ids)))
t0 = time.time()
Xtr = load('train_images', tr[fcol].tolist()); tfn = te.set_index(te.columns[0])[te.columns[1]].reindex(ss[ss.columns[0]]).tolist(); print(tfn[:3]); Xte = load('test_images', tfn)
classes = sorted(tr[lc].unique().tolist()); cmap = {c: i for i, c in enumerate(classes)}
ytr = tr[lc].map(cmap).values
print('loaded', Xtr.shape, Xte.shape, f'{time.time()-t0:.0f}s', classes)
print('train mean/std', Xtr.mean((0,1,2)), Xtr.std((0,1,2)), '| test mean/std', Xte.mean((0,1,2)), Xte.std((0,1,2)))
mean = torch.tensor(Xtr.mean((0,1,2)) / 255, device=dev).view(1,3,1,1).float(); std = torch.tensor(Xtr.std((0,1,2)) / 255, device=dev).view(1,3,1,1).float()
g = lambda a: torch.from_numpy(a).permute(0,3,1,2).contiguous().to(dev)  # uint8 NCHW on GPU
rng = np.random.RandomState(0); perm = rng.permutation(len(Xtr)); nh = CFG['holdout']
hi, ti = perm[:nh], perm[nh:]
Xt, Yt = g(Xtr[ti]), torch.from_numpy(ytr[ti]).to(dev); Xh, Yh = g(Xtr[hi]), torch.from_numpy(ytr[hi]).to(dev); Xe = g(Xte)

# ---------- GPU augmentation ----------
def crop(x, p=4):
    N, C, H, W = x.shape; xp = F.pad(x, (p,p,p,p), mode='reflect')
    i = torch.randint(0, 2*p+1, (N,), device=x.device); j = torch.randint(0, 2*p+1, (N,), device=x.device); ar = torch.arange(H, device=x.device)
    r = (i[:,None] + ar)[:,None,:,None]; c = (j[:,None] + ar)[:,None,None,:]
    return xp[torch.arange(N, device=x.device)[:,None,None,None], torch.arange(C, device=x.device)[None,:,None,None], r, c]
def flip(x): return torch.where(torch.rand(x.shape[0],1,1,1,device=x.device) < .5, x.flip(3), x)
def U(n, lo, hi_, dv): return torch.rand(n,1,1,1,device=dv) * (hi_ - lo) + lo
def box_blur(x): return F.avg_pool2d(F.pad(x, (1,1,1,1), mode='reflect'), 3, 1)
def robust(x, s):  # x in [0,1]; s = strength 0..1 ; random photometric / noise / blur shifts
    N, dv = x.shape[0], x.device
    m = lambda p: (torch.rand(N,1,1,1,device=dv) < p * s).float()
    x = x * (1 + m(.6) * U(N,-.35,.35,dv))                                   # brightness
    mu = x.mean((1,2,3), keepdim=True); x = (x - mu) * (1 + m(.6) * U(N,-.45,.45,dv)) + mu   # contrast
    gr = x.mean(1, keepdim=True); x = gr + (x - gr) * (1 + m(.5) * U(N,-.7,.5,dv))          # saturation
    b = m(.35); x = b * (0.5*x + 0.5*box_blur(x)) + (1-b) * x if s > 0 else x               # blur
    x = x + m(.4) * torch.randn_like(x) * U(N,0,.09,dv)                                     # gaussian noise
    return x.clamp(0, 1)
def erase(x, p=.5, sz=10):
    N, C, H, W = x.shape; dv = x.device
    cy = torch.randint(0, H, (N,1,1), device=dv); cx = torch.randint(0, W, (N,1,1), device=dv)
    yy = torch.arange(H, device=dv).view(1,H,1); xx = torch.arange(W, device=dv).view(1,1,W)
    msk = ((yy - cy).abs() < sz//2) & ((xx - cx).abs() < sz//2) & (torch.rand(N,1,1,device=dv) < p)
    return torch.where(msk[:,None], torch.zeros_like(x), x)
def norm(x):
    if CFG.get('inorm'): x = (x - x.mean((1,2,3), keepdim=True)) / (x.std((1,2,3), keepdim=True) + 1e-2)  # per-image standardisation: invariant to brightness/contrast shift
    else: x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last)
def cutmix(x, y, nc):
    N, _, H, W = x.shape; lam = np.random.beta(1., 1.); idx = torch.randperm(N, device=x.device)
    rh, rw = int(H * math.sqrt(1 - lam)), int(W * math.sqrt(1 - lam)); cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2, x1, x2 = max(cy - rh//2, 0), min(cy + rh//2, H), max(cx - rw//2, 0), min(cx + rw//2, W)
    x = x.clone(); x[:, :, y1:y2, x1:x2] = x[idx][:, :, y1:y2, x1:x2]; lam = 1 - (y2-y1)*(x2-x1)/(H*W)
    return x, y, y[idx], lam

# ---------- model: ResNet (basic blocks, pre-activation style stages), <5M params ----------
class Block(nn.Module):
    def __init__(s, i, o, st):
        super().__init__()
        s.c1 = nn.Conv2d(i, o, 3, st, 1, bias=False); s.b1 = nn.BatchNorm2d(o)
        s.c2 = nn.Conv2d(o, o, 3, 1, 1, bias=False); s.b2 = nn.BatchNorm2d(o)
        s.sk = None if (st == 1 and i == o) else nn.Sequential(nn.Conv2d(i, o, 1, st, bias=False), nn.BatchNorm2d(o))
    def forward(s, x):
        y = F.relu(s.b1(s.c1(x)), inplace=True); y = s.b2(s.c2(y))
        return F.relu(y + (x if s.sk is None else s.sk(x)), inplace=True)
class Net(nn.Module):
    def __init__(s, w=(64,128,256), nb=(3,3,3), nc=10, drop=0.):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv2d(3, w[0], 3, 1, 1, bias=False), nn.BatchNorm2d(w[0]), nn.ReLU(inplace=True))
        L, i = [], w[0]
        for k, (c, n) in enumerate(zip(w, nb)):
            for b in range(n): L.append(Block(i, c, 2 if (b == 0 and k > 0) else 1)); i = c
        s.layers = nn.Sequential(*L); s.drop = nn.Dropout(drop); s.fc = nn.Linear(i, nc)
    def forward(s, x): return s.fc(s.drop(F.adaptive_avg_pool2d(s.layers(s.stem(x)), 1).flatten(1)))
model = Net(CFG['w'], CFG['nb'], len(classes), CFG['drop']).to(dev).to(memory_format=torch.channels_last)
nparam = sum(p.numel() for p in model.parameters() if p.requires_grad); print('trainable params', nparam); assert nparam <= 5_000_000
ema = torch.optim.swa_utils.AveragedModel(model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(CFG['ema']), use_buffers=True)

# ---------- training ----------
E_, bs = CFG['epochs'], CFG['bs']; spe = len(Xt) // bs; total = E_ * spe
decay, no_decay = [], []
for n_, p in model.named_parameters(): (no_decay if p.ndim <= 1 else decay).append(p)
opt = torch.optim.SGD([{'params': decay, 'weight_decay': CFG['wd']}, {'params': no_decay, 'weight_decay': 0}], lr=CFG['lr'], momentum=.9, nesterov=True)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], total_steps=total, pct_start=.15, anneal_strategy='cos', div_factor=20, final_div_factor=500)
scaler = torch.cuda.amp.GradScaler(); crit = nn.CrossEntropyLoss(label_smoothing=CFG['ls'])
@torch.no_grad()
def predict(m, X, tta=True, bsz=1000):
    m.eval(); out = []
    for i in range(0, len(X), bsz):
        x = norm(X[i:i+bsz].float() / 255)
        with torch.autocast('cuda', dtype=torch.float16):
            lg = F.softmax(m(x).float(), 1)
            if tta: lg = lg + F.softmax(m(x.flip(3)).float(), 1); lg = lg / 2
        out.append(lg)
    return torch.cat(out)
def noisy_holdout():
    g_ = torch.Generator(device=dev); torch.manual_seed(123)
    return (robust(Xh.float() / 255, 1.0) * 255).round()
Xhn = noisy_holdout() if nh else None
def acc(m, X, Y): return (predict(m, X).argmax(1) == Y).float().mean().item()
t0 = time.time(); step = 0
for ep in range(E_):
    model.train(); pm = torch.randperm(len(Xt), device=dev); tl = 0.
    for k in range(spe):
        ix = pm[k*bs:(k+1)*bs]; x = Xt[ix].float() / 255; y = Yt[ix]
        x = flip(crop(x)); x = robust(x, CFG['rob']) if CFG['rob'] > 0 else x
        if CFG['erase'] > 0: x = erase(x, CFG['erase'])
        x = norm(x)
        with torch.autocast('cuda', dtype=torch.float16):
            if np.random.rand() < CFG['cutmix']:
                x, ya, yb, lam = cutmix(x, y, 10); o = model(x); loss = lam * crit(o, ya) + (1 - lam) * crit(o, yb)
            else: loss = crit(model(x), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.); scaler.step(opt); scaler.update(); sched.step(); ema.update_parameters(model); tl += loss.item() if k % 20 == 0 else 0
    if ep % 5 == 4 or ep == E_ - 1 or ep < 2:
        msg = f'ep {ep+1}/{E_} loss {tl/(spe/20):.3f} t {time.time()-t0:.0f}s'
        if nh: msg += f' | holdout clean raw {acc(model,Xh,Yh):.4f} ema {acc(ema.module,Xh,Yh):.4f} | shifted ema {acc(ema.module,Xhn,Yh):.4f}'
        print(msg, flush=True)
    else: print(f'ep {ep+1} t {time.time()-t0:.0f}s', flush=True)

# ---------- predict & write submission ----------
fm = ema.module; P = predict(fm, Xe).cpu().numpy(); np.save('/kaggle/working/test_probs.npy', P)
sub = ss.copy(); sub[ss.columns[1]] = [classes[i] for i in P.argmax(1)]; sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.head(), sub[ss.columns[1]].value_counts().to_dict()); print('DONE params', nparam)


(50000, 3) (2000, 2) (2000, 2) ['id', 'filename', 'label'] ['id', 'label']
['test_0000.png', 'test_0001.png', 'test_0002.png']
loaded (50000, 32, 32, 3) (2000, 32, 32, 3) 23s [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
train mean/std [125.30691805 122.95039414 113.86538318] [62.99321928 62.08870764 66.70489964] | test mean/std [130.80530273 126.93861768 117.61218408] [67.53519388 66.86670643 70.14086543]
trainable params 4327754


/tmp/ipykernel_23/3146039900.py:99: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(); crit = nn.CrossEntropyLoss(label_smoothing=CFG['ls'])


ep 1/80 loss 2.090 t 24s | holdout clean raw 0.5765 ema 0.1115 | shifted ema 0.1115
ep 2/80 loss 1.702 t 39s | holdout clean raw 0.5540 ema 0.1115 | shifted ema 0.1115
ep 3 t 55s
ep 4 t 69s
ep 5/80 loss 1.445 t 84s | holdout clean raw 0.7355 ema 0.1940 | shifted ema 0.1930
ep 6 t 101s
ep 7 t 117s
ep 8 t 132s
ep 9 t 147s
ep 10/80 loss 1.354 t 162s | holdout clean raw 0.7705 ema 0.5460 | shifted ema 0.4985
ep 11 t 178s
ep 12 t 193s
ep 13 t 209s
ep 14 t 224s
ep 15/80 loss 1.553 t 239s | holdout clean raw 0.7875 ema 0.8430 | shifted ema 0.8005
ep 16 t 255s
ep 17 t 270s
ep 18 t 285s
ep 19 t 301s
ep 20/80 loss 1.203 t 316s | holdout clean raw 0.7995 ema 0.8875 | shifted ema 0.8540
ep 21 t 332s
ep 22 t 347s
ep 23 t 363s
ep 24 t 378s
ep 25/80 loss 1.188 t 393s | holdout clean raw 0.8520 ema 0.8975 | shifted ema 0.8735
ep 26 t 409s
ep 27 t 424s
ep 28 t 439s
ep 29 t 454s
ep 30/80 loss 1.001 t 470s | holdout clean raw 0.8735 ema 0.9110 | shifted ema 0.8865
ep 31 t 486s
ep 32 t 501s
ep 33 t 516s
e